# 1: Define Group patients by similar features

## Plain English summary

This notebook defines many different "types" of patients. Each group of patients has similar properties, and for each group we look at all nine of the features that will be used in the machine learning model. This notebook defines 576 different groups.

The groups are not divided up too finely otherwise there would not be very many people in any group.
Also, two of the times in the patient data have been combined into one new value. We use onset-to-scan time instead of separate onset-to-arrival and arrival-to-scan times. If they were kept separate, there would be too few patients in any group.

The patients are grouped by:

| Feature | Options |
| --- | --- |
| Stroke severity | Mild, moderate, severe |
| Prior disability | Mild (mRS 0 or 1), moderate (mRS 2 or 3), severe (mRS 4 or 5) |
| Age | Below 80, 80 or higher |
| Infarction | Yes, no |
| Onset to _scan_ time | Below four hours, at least four hours |
| Precise onset known | Yes, no |
| Onset during sleep | Yes, no |
| Afib anticoagulants | Yes, no |

All of the different combinations of these features make $3 \times 3 \times 2 \times 2 \times 2 \times 2 \times 2 \times 2  = 576$ groups.

## Load imports

In [1]:
import pandas as pd
import numpy as np

from dataclasses import dataclass

# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

## Set up paths and filenames

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    data_save_path: str = '../../../stroke_utilities/data'
    notebook: str = ''

paths = Paths()

## Define groups

Set up the groups by defining how many options there are for each feature. They are called "masks" because the feature options will be used to mask out unwanted patients from the full dataset.

In [3]:
# How many masks are in each category?
masks = {
    'onset_scan':2,
    'severity':3,
    'mrs':3,
    'age':2,
    'infarction':2,
    'precise':2,
    'sleep':2,
    'anticoag':2
}
masks_names = list(masks.keys())
masks_lens = list(masks.values())

Also store a way to convert the number labels back to more meaningful labels:

In [4]:
mask_str_dict = {
    'onset_scan':{0:'<=4hr', 1:'>4hr'},
    'severity':{0:'Mild', 1:'Moderate', 2:'Severe'},
    'mrs':{0:'0 to 1', 1:'2 to 3', 2:'4 to 5'},
    'age':{0:'Below 80', 1:'At least 80'},
    'infarction':{0:'No', 1:'Yes'},
    'precise':{0:'No', 1:'Yes'},
    'sleep':{0:'No', 1:'Yes'},
    'anticoag':{0:'No', 1:'Yes'},
    }

Name each feature option as a number (option 0, 1, 2...). Find every unique combination of these feature options and store the lists of numbers.

In [5]:
# This could be written more compactly but it works for now:
inds_lists = []

for a in range(masks_lens[0]):
    for b in range(masks_lens[1]):
        for c in range(masks_lens[2]):
            for d in range(masks_lens[3]):
                for e in range(masks_lens[4]):
                    for f in range(masks_lens[5]):
                        for g in range(masks_lens[6]):
                            for h in range(masks_lens[7]):
                                inds_lists.append([a, b, c, d, e, f, g, h])

The first few combinations look like this:

In [6]:
inds_lists[:8]

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 1],
 [0, 0, 0, 0, 0, 1, 1, 0],
 [0, 0, 0, 0, 0, 1, 1, 1]]

Place these lists of lists into a dataframe so that we can label which number belongs to which feature. Also create a new column, `mask_number`, to label each unique combination of the features.

In [7]:
df_mask_numbers = pd.DataFrame(inds_lists, columns=[m + '_mask_number' for m in masks_names])

df_mask_numbers['mask_number'] = np.arange(len(df_mask_numbers))

df_mask_numbers

,onset_scan_mask_number,severity_mask_number,mrs_mask_number,age_mask_number,infarction_mask_number,precise_mask_number,sleep_mask_number,anticoag_mask_number,mask_number
0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,1
2,0,0,0,0,0,0,1,0,2
3,0,0,0,0,0,0,1,1,3
4,0,0,0,0,0,1,0,0,4
...,...,...,...,...,...,...,...,...,...
571,1,2,2,1,1,0,1,1,571
572,1,2,2,1,1,1,0,0,572
573,1,2,2,1,1,1,0,1,573
574,1,2,2,1,1,1,1,0,574


Save the mask combinations and their labels to file:

In [8]:
df_mask_numbers.to_csv('mask_numbers.csv', index=False)

Translate the numbers back to strings for a better look:

In [9]:
df_mask_labels = df_mask_numbers.copy()

# Remove the _mask_number part of the columns:
df_mask_labels = df_mask_labels.rename(
    columns=dict(zip([m + '_mask_number' for m in masks_names], masks_names))
)

# Replace numbers with labels:
df_mask_labels = df_mask_labels.replace(mask_str_dict)

df_mask_labels

,onset_scan,severity,mrs,age,infarction,precise,sleep,anticoag,mask_number
0,<=4hr,Mild,0 to 1,Below 80,No,No,No,No,0
1,<=4hr,Mild,0 to 1,Below 80,No,No,No,Yes,1
2,<=4hr,Mild,0 to 1,Below 80,No,No,Yes,No,2
3,<=4hr,Mild,0 to 1,Below 80,No,No,Yes,Yes,3
4,<=4hr,Mild,0 to 1,Below 80,No,Yes,No,No,4
...,...,...,...,...,...,...,...,...,...
571,>4hr,Severe,4 to 5,At least 80,Yes,No,Yes,Yes,571
572,>4hr,Severe,4 to 5,At least 80,Yes,Yes,No,No,572
573,>4hr,Severe,4 to 5,At least 80,Yes,Yes,No,Yes,573
574,>4hr,Severe,4 to 5,At least 80,Yes,Yes,Yes,No,574
